<div style="background-color:#000;"><img src="pqn.png"></img></div><div><a href="https://pyquantnews.com/">PyQuant News</a> is where finance practitioners level up with Python for quant finance, algorithmic trading, and market data analysis. Looking to get started? Check out the fastest growing, top-selling course to <a href="https://www.pyquantnews.com/getting-started-with-python-for-quant-finance/">get started with Python for quant finance</a>. For educational purposes. Not investment advice. Use at your own risk.</div>

## Library installation

Install the one package this notebook needs. OpenBB pulls the share prices through its default equity provider and the US inflation series from FRED, so we don't need a separate economic data library.

In [ ]:
!pip install openbb

The CPI request routes to FRED, which needs the FRED extension installed and a free FRED API key saved to your OpenBB account before the call returns rows. An empty frame here almost always means the key isn't registered, not that the arguments are wrong.

## Imports and setup

We import obb from openbb, which covers the two endpoints used below, obb.stocks.load and obb.economy.cpi. Each one returns an OBBject rather than a DataFrame, and .to_dataframe() converts it into a pandas DataFrame with a date index.

In [1]:
from openbb import obb

## Pull BMW prices for one year

Pull daily closing prices for WMT, Walmart, over a window that starts in January 2025.

In [34]:
stocks_dataframe = obb.equity.price.historical(
    "WMT",
    start_date="2025-01-01",
    end_date="2026-10-01",
).to_dataframe()

Pair a CPI reading from a different month with this twelve month price change and you subtract one period's inflation from another period's gain, and nothing in the output flags it.

Take the first and last closing price present in the result, which are the first and last trading days inside the window rather than the calendar dates themselves, and turn them into a percentage gain.

In [40]:
price_start = stocks_dataframe["close"].iloc[0]
price_end = stocks_dataframe["close"].iloc[-1]
nominal_returns = (price_end - price_start) / price_start
print(f"Nominal price returns: {nominal_returns}")

Nominal price returns: 0.15844446818033855


This is the nominal return, the raw percentage change in price before we account for anything else. A brokerage statement may report a different figure for the same position, because account performance folds in cash flows and dividends.

## Request US inflation over the same window

Ask OpenBB for US CPI with units set to "growth_same", which returns each month's percentage change against the same month one year earlier, so the final row is the most recent monthly observation.

In [44]:
inflation = obb.economy.cpi(
    ["united_states"],
    units="growth_same",
    start_date="2025-01-01",
    end_date="2026-10-01",
    frequency="monthly",
).to_dataframe()

"growth_same" requests year over year percentage growth, so a single row already carries a twelve month inflation rate and we're not stitching monthly numbers together ourselves. That rate lines up with the twelve month price change above only when the final CPI month falls inside the same window, which is why both calls use the same dates.

In [45]:
inflation = inflation["value"].iloc[-1]
print(f"Inflation: {inflation}")

Inflation: 0.03737684729064075


obb.economy.cpi accepts a list of countries, so the returned frame carries "united_states" as the outer column level sitting above "value". We divide by 100 because the value field arrives in percentage points, like 6.0, while nominal_returns is already a decimal, like 0.06. Skip the division and the real return comes out off by a factor of a hundred.

## Convert the gain into real terms

Apply the Fisher relationship to divide out the change in purchasing power and get the real return.

In [46]:
real_returns = (1 + nominal_returns) / (1 + inflation) - 1
print(f"Real returns: {real_returns}")

Real returns: 0.116705535896521


We divide because the Fisher relation compares the gross nominal return, 1 plus the gain, against the price level ratio, 1 plus inflation. Subtraction is close when inflation sits near 2%, and the error grows with the inflation rate times the gap between the two numbers.

<a href="https://pyquantnews.com/">PyQuant News</a> is where finance practitioners level up with Python for quant finance, algorithmic trading, and market data analysis. Looking to get started? Check out the fastest growing, top-selling course to <a href="https://www.pyquantnews.com/getting-started-with-python-for-quant-finance/">get started with Python for quant finance</a>. For educational purposes. Not investment advice. Use at your own risk.